In [ ]:
import anndata as ad

adata = ad.read_h5ad("./data/B_cell/IGVFFI3928IUMP.h5ad")
adata

In [ ]:
import scanpy as sc

sc.pp.filter_cells(adata, min_counts=1000)
sc.pp.filter_cells(adata, min_genes=500)

sc.pp.filter_genes(adata, min_cells=20)
adata

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
adata.layers["spliced"] = adata.layers["mature"]
adata.layers["unspliced"] = adata.layers["nascent"]

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
)

adata = adata[:, adata.var["highly_variable"]].copy()

sc.pp.scale(adata)
sc.tl.pca(adata)

In [ ]:
import mygene
import pandas as pd
import numpy as np

def clean_gene_names(adata):
    mg = mygene.MyGeneInfo()
    
    # 1. Prepare query list (clean versioning like .1, .2)
    original_ids = adata.var_names.tolist()
    clean_ids = [str(g).split('.')[0] for g in original_ids]
    
    # 2. Query as a list of dicts (more robust than the dataframe output)
    print(f"Querying {len(clean_ids)} genes...")
    results = mg.querymany(
        clean_ids, 
        scopes="ensembl.gene", 
        fields="symbol", 
        species="human", 
        as_dataframe=False, # Use list of dicts to avoid index/KeyErrors
        verbose=False
    )
    
    # 3. Build a strict mapping: Only add if 'symbol' actually exists
    symbol_map = {}
    for item in results:
        query_id = item.get('query')
        symbol = item.get('symbol')
        if query_id and symbol and str(symbol).lower() != 'nan':
            symbol_map[query_id] = str(symbol)

    # 4. Final assignment: Map it or Keep it
    new_names = []
    for i, orig in enumerate(original_ids):
        clean = clean_ids[i]
        # Priority: 1. Found Symbol, 2. Cleaned ID, 3. Original string
        final_name = symbol_map.get(clean, clean if str(clean).lower() != 'nan' else orig)
        new_names.append(final_name)
    
    # 5. Update AnnData
    adata.var['original_id'] = original_ids # Keep a backup just in case
    adata.var_names = new_names
    adata.var_names_make_unique()
    
    print(f"Mapping complete. Unique names assigned.")
    return adata

# Execution
adata = clean_gene_names(adata)

In [ ]:
import scvelo as scv

scv.pp.moments(adata, n_neighbors=30, n_pcs=30)
scv.tl.velocity(adata, mode="stochastic")
scv.tl.velocity_graph(adata)

In [ ]:
sc.tl.umap(adata)

scv.pl.velocity_embedding_stream(
    adata,
    basis="umap"
)

In [ ]:
# Adjusting parameters for a tighter layout
sc.tl.umap(adata, spread=0.8, min_dist=0.7)

# Plotting the velocity stream
scv.pl.velocity_embedding_stream(
    adata, 
    basis="umap"
)

In [ ]:
scv.tl.recover_dynamics(adata)
scv.tl.latent_time(adata)

In [ ]:
adata.var_names

In [ ]:
adata.write("./data/B_cell/bcell_velocity_processed_pause.h5ad")

In [ ]:
# ============================================================
# Reload RAW (full gene space)
# ============================================================
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd

adata = ad.read_h5ad("./data/B_cell/bcell_velocity_processed_pause.h5ad")
adata_raw = ad.read_h5ad("./data/B_cell/IGVFFI3928IUMP.h5ad")

# same basic filtering (cells only, DO NOT subset genes aggressively)
sc.pp.filter_cells(adata_raw, min_counts=1000)
sc.pp.filter_cells(adata_raw, min_genes=500)

# normalize + log (same as tutorial)
sc.pp.normalize_total(adata_raw, target_sum=1e4)
sc.pp.log1p(adata_raw)

# ============================================================
# Convert Ensembl → gene symbols (for adata_raw)
# ============================================================
import mygene

mg = mygene.MyGeneInfo()

original_ids = adata_raw.var_names.tolist()
clean_ids = [str(g).split('.')[0] for g in original_ids]

print("Mapping genes for cell cycle scoring...")

res = mg.querymany(
    clean_ids,
    scopes="ensembl.gene",
    fields="symbol",
    species="human",
    as_dataframe=False,
    verbose=False
)

symbol_map = {}
for item in res:
    q = item.get("query")
    s = item.get("symbol")
    if q and s and str(s).lower() != "nan":
        symbol_map[q] = str(s)

# assign symbols (fallback to Ensembl if missing)
new_names = []
for i, orig in enumerate(original_ids):
    clean = clean_ids[i]
    new_names.append(symbol_map.get(clean, clean))

adata_raw.var["original_id"] = original_ids
adata_raw.var_names = new_names
adata_raw.var_names_make_unique()

print("Gene mapping done.")

# ============================================================
# Cell cycle gene list (Scanpy tutorial)
# ============================================================
import urllib.request

url = "https://raw.githubusercontent.com/theislab/scanpy_usage/master/180209_cell_cycle/data/regev_lab_cell_cycle_genes.txt"
urllib.request.urlretrieve(url, "cell_cycle_genes.txt")

cell_cycle_genes = [x.strip() for x in open("cell_cycle_genes.txt")]

s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]

# filter to genes that exist
s_genes = [g for g in s_genes if g in adata_raw.var_names]
g2m_genes = [g for g in g2m_genes if g in adata_raw.var_names]


# ============================================================
# Compute cell cycle
# ============================================================
sc.tl.score_genes_cell_cycle(
    adata_raw,
    s_genes=s_genes,
    g2m_genes=g2m_genes
)


# ============================================================
# Transfer scores to your processed adata
# ============================================================
common_cells = adata.obs_names.intersection(adata_raw.obs_names)

adata.obs.loc[common_cells, "S_score"]   = adata_raw.obs.loc[common_cells, "S_score"]
adata.obs.loc[common_cells, "G2M_score"] = adata_raw.obs.loc[common_cells, "G2M_score"]
adata.obs.loc[common_cells, "phase"]     = adata_raw.obs.loc[common_cells, "phase"]

# optional composite
adata.obs["cycle_score"] = (
    adata.obs["S_score"] + adata.obs["G2M_score"]
)

print("Cell cycle scores added.")

# ============================================================
# Add p21 / p27 expression (G0 markers)
# ============================================================
genes = {
    "p21": "CDKN1A",
    "p27": "CDKN1B"
}

for key, gene in genes.items():
    
    if gene not in adata_raw.var_names:
        print(f"{gene} not found in adata_raw")
        continue
    
    x = adata_raw[:, gene].X
    
    # handle sparse
    if hasattr(x, "toarray"):
        x = x.toarray().flatten()
    else:
        x = np.array(x).flatten()
    
    # align to processed adata
    aligned = np.full(adata.n_obs, np.nan)
    
    raw_idx = adata_raw.obs_names.get_indexer(common_cells)
    proc_idx = adata.obs_names.get_indexer(common_cells)
    
    aligned[proc_idx] = x[raw_idx]
    
    adata.obs[key] = aligned

print("p21 / p27 added to adata.obs")

adata.write("./data/B_cell/bcell_velocity_processed_pause.h5ad")

In [ ]:
adata

In [ ]:
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd

adata_raw = ad.read_h5ad("./data/B_cell/IGVFFI3928IUMP.h5ad")

# same basic filtering (cells only, DO NOT subset genes aggressively)
sc.pp.filter_cells(adata_raw, min_counts=1000)
sc.pp.filter_cells(adata_raw, min_genes=500)

# normalize + log (same as tutorial)
sc.pp.normalize_total(adata_raw, target_sum=1e4)
sc.pp.log1p(adata_raw)

# ============================================================
# Convert Ensembl → gene symbols (for adata_raw)
# ============================================================
import mygene

mg = mygene.MyGeneInfo()

original_ids = adata_raw.var_names.tolist()
clean_ids = [str(g).split('.')[0] for g in original_ids]

print("Mapping genes for cell cycle scoring...")

res = mg.querymany(
    clean_ids,
    scopes="ensembl.gene",
    fields="symbol",
    species="human",
    as_dataframe=False,
    verbose=False
)

symbol_map = {}
for item in res:
    q = item.get("query")
    s = item.get("symbol")
    if q and s and str(s).lower() != "nan":
        symbol_map[q] = str(s)

# assign symbols (fallback to Ensembl if missing)
new_names = []
for i, orig in enumerate(original_ids):
    clean = clean_ids[i]
    new_names.append(symbol_map.get(clean, clean))

adata_raw.var["original_id"] = original_ids
adata_raw.var_names = new_names
adata_raw.var_names_make_unique()

print("Gene mapping done.")

adata_raw

In [ ]:
# Optional: re-compute neighbors if unsure
sc.pp.neighbors(adata_raw, n_neighbors=15, n_pcs=30)

# Leiden clustering
sc.tl.leiden(adata_raw, resolution=0.5)  # adjust later if needed

# Visualize
sc.tl.umap(adata_raw)
sc.pl.umap(adata_raw, color='leiden')

In [ ]:
markers = {
    "Naive": ["TCL1A", "IL4R", "FCER2", "IGHD", "IGHM", "MS4A1"],
    "ActB": ["CD69", "CD83", "FOS", "JUN", "EGR1", "CD40"],
    "preGCBC": ["BCL6", "MEF2B", "CD83", "HLA-DRA"],
    "prePB": ["IRF4", "PRDM1", "XBP1", "JCHAIN", "CD27"],
    "PB": ["PRDM1", "XBP1", "JCHAIN", "MZB1", "SDC1", "TNFRSF17"]
}

markers_filtered = {}

for key, gene_list in markers.items():
    genes_present = [g for g in gene_list if g in adata_raw.var_names]
    if len(genes_present) > 0:
        markers_filtered[key] = genes_present

markers_filtered

In [ ]:
sc.pl.dotplot(
    adata_raw,
    markers_filtered,
    groupby='leiden',
    standard_scale='var'
)

In [ ]:
cluster_map = {
    "0": "prePB",
    "1": "prePB",
    "2": "PB",
    "3": "PB",
    "4": "Naive",
    "5": "PB",
    "6": "Naive",
    "7": "ActB",
    "8": "preGCBC",
    "9": "Naive",
    "10": "ActB",
    "11": "preGCBC",
    "12": "PB",
}

adata_raw.obs["cell_type"] = adata_raw.obs["leiden"].map(cluster_map)
adata_raw.obs["cell_type"].value_counts()

In [ ]:
adata.obs["cell_type"] = adata_raw.obs["cell_type"]
sc.pl.umap(
    adata_raw,
    color=["leiden", "cell_type"],
    legend_loc="on data",
    frameon=False
)